# Phase 3 — 数据采集与信号处理
# Data Collection & Signal Processing Pipeline

**EchoGlove Edge-AI Data Glove V3**

本文档覆盖：
1. 传感器数据格式与特征向量定义
2. 卡尔曼滤波器（Kalman Filter）实现与公式推导
3. Min-Max 特征归一化
4. 滑动窗口构建（Sliding Window）
5. 合成数据生成（开发阶段替代真实采集）
6. Edge Impulse 数据转发器 CSV 格式
7. 数据可视化与质量验证

---

## 硬件拓扑

```
ESP32-S3  ──I2C──→  TCA9548A (Mux)
                       ├─ CH0: TMAG5273 (Thumb)   → XYZ mT
                       ├─ CH1: TMAG5273 (Index)    → XYZ mT
                       ├─ CH2: TMAG5273 (Middle)   → XYZ mT
                       ├─ CH3: TMAG5273 (Ring)     → XYZ mT
                       ├─ CH4: TMAG5273 (Pinky)    → XYZ mT
                       └─ CH5: BNO085 IMU          → Quat + Gyro
```

## 1. 环境准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import json
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import Tuple, List

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# 项目根目录
PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'CLAUDE.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'Project root: {PROJECT_ROOT}')

## 2. 常量定义与特征向量

### 2.1 编译时常量（对应 `data_structures.h`）

| 常量 | 值 | 说明 |
|------|-----|------|
| `NUM_HALL_SENSORS` | 5 | TMAG5273 三轴霍尔传感器数量 |
| `HALL_AXES` | 3 | 每个传感器的轴数 (X, Y, Z) |
| `HALL_FEATURE_COUNT` | 15 | 5 × 3 = 15 个霍尔特征 |
| `IMU_FEATURE_COUNT` | 6 | 3 Euler + 3 Gyro |
| `FEATURE_COUNT` | 21 | 每帧总特征数 |
| `WINDOW_SIZE` | 30 | 滑动窗口帧数 |
| `SENSOR_RATE_HZ` | 100 | 采样率 |
| `NUM_CLASSES` | 46 | 手势类别数 |

### 2.2 特征向量布局

每帧特征向量 $\mathbf{x}_t \in \mathbb{R}^{21}$ 的排列：

$$
\mathbf{x}_t = [\underbrace{h_0^x, h_0^y, h_0^z, \ldots, h_4^x, h_4^y, h_4^z}_{15 \text{ Hall}}, \underbrace{\phi, \theta, \psi}_{3 \text{ Euler}}, \underbrace{\omega_x, \omega_y, \omega_z}_{3 \text{ Gyro}}]
$$

其中：
- $h_i^{x,y,z}$: 第 $i$ 个霍尔传感器的三轴磁场强度（mT）
- $\phi, \theta, \psi$: 手腕姿态欧拉角（roll, pitch, yaw，单位：度）
- $\omega_{x,y,z}$: 角速度（deg/s）

In [ ]:
# ============================================================
# 常量定义（与 data_structures.h 保持一致）
# ============================================================

NUM_HALL_SENSORS = 5
HALL_AXES = 3
HALL_FEATURE_COUNT = NUM_HALL_SENSORS * HALL_AXES  # 15
IMU_FEATURE_COUNT = 6  # 3 euler + 3 gyro
FEATURE_COUNT = HALL_FEATURE_COUNT + IMU_FEATURE_COUNT  # 21
WINDOW_SIZE = 30
SENSOR_RATE_HZ = 100
NUM_CLASSES = 46

# 特征通道名称
FEATURE_NAMES = []
for i in range(NUM_HALL_SENSORS):
    for axis in ['x', 'y', 'z']:
        FEATURE_NAMES.append(f'hall_{i}_{axis}')
FEATURE_NAMES += ['euler_roll', 'euler_pitch', 'euler_yaw']
FEATURE_NAMES += ['gyro_x', 'gyro_y', 'gyro_z']

assert len(FEATURE_NAMES) == FEATURE_COUNT, f'Expected {FEATURE_COUNT}, got {len(FEATURE_NAMES)}'

# 加载手势标签
labels_path = PROJECT_ROOT / 'glove_relay' / 'data' / 'gesture_labels.json'
with open(labels_path) as f:
    GESTURE_LABELS = json.load(f)

print(f'特征维度: {FEATURE_COUNT}')
print(f'通道名称 ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')
print(f'手势类别数: {len(GESTURE_LABELS)}')
print(f'示例标签: {GESTURE_LABELS[0]}')  # {"id": 0, "name_cn": "零", "name_en": "zero"}

## 3. 卡尔曼滤波器（Kalman Filter）

### 3.1 算法原理

项目使用 **一维恒值模型卡尔曼滤波器**，独立作用于每个传感器通道（共 21 个实例）。

#### 状态空间模型

状态转移方程（无运动模型，恒值假设）：

$$
\hat{x}_{k|k-1} = \hat{x}_{k-1|k-1} \quad \text{(预测 = 上次估计)}
$$

$$
P_{k|k-1} = P_{k-1|k-1} + Q \quad \text{(先验协方差)}
$$

#### 更新步骤

卡尔曼增益：

$$
K_k = \frac{P_{k|k-1}}{P_{k|k-1} + R}
$$

状态更新：

$$
\hat{x}_{k|k} = \hat{x}_{k|k-1} + K_k (z_k - \hat{x}_{k|k-1})
$$

协方差更新：

$$
P_{k|k} = (1 - K_k) \cdot P_{k|k-1}
$$

#### 参数

| 参数 | 值 | 含义 |
|------|-----|------|
| $Q$ | 0.001 | 过程噪声方差（模型不确定度） |
| $R$ | 0.01 | 测量噪声方差（传感器精度） |

**$Q/R$ 比值** 决定了滤波器的信任分配：
- $Q/R \ll 1$：更多信任模型（强平滑，响应慢）
- $Q/R \gg 1$：更多信任测量（弱平滑，响应快）
- 本项目 $Q/R = 0.1$，偏向强平滑

In [ ]:
class KalmanFilter1D:
    """
    一维恒值模型卡尔曼滤波器。
    对应固件 lib/Filters/KalmanFilter1D.h 的 Python 实现。
    
    Parameters
    ----------
    Q : float
        过程噪声方差 (process noise)
    R : float
        测量噪声方差 (measurement noise)
    """
    
    def __init__(self, Q: float = 0.001, R: float = 0.01):
        self.Q = Q
        self.R = R
        self.x = 0.0  # 状态估计
        self.P = 1.0  # 估计协方差
        self._initialized = False
    
    def update(self, z: float) -> float:
        """
        输入新的测量值 z，返回滤波后的估计值。
        
        Parameters
        ----------
        z : float
            当前时刻的测量值
            
        Returns
        -------
        float
            滤波后的状态估计
        """
        if not self._initialized:
            # 首次测量：直接用测量值初始化
            self.x = z
            self.P = self.R
            self._initialized = True
            return self.x
        
        # ---- 预测 (Predict) ----
        x_pred = self.x                    # x_{k|k-1} = x_{k-1|k-1}
        P_pred = self.P + self.Q           # P_{k|k-1} = P_{k-1|k-1} + Q
        
        # ---- 更新 (Update) ----
        K = P_pred / (P_pred + self.R)     # 卡尔曼增益
        self.x = x_pred + K * (z - x_pred) # 状态更新
        self.P = (1 - K) * P_pred          # 协方差更新
        
        return self.x
    
    def reset(self):
        """重置滤波器状态。"""
        self.x = 0.0
        self.P = 1.0
        self._initialized = False


# ---- 验证：对比不同 Q/R 参数 ----
np.random.seed(42)
t = np.linspace(0, 2, 200)  # 2秒，100Hz
true_signal = np.sin(2 * np.pi * 1.5 * t)  # 1.5Hz 正弦波
noise = np.random.normal(0, 0.15, len(t))
noisy = true_signal + noise

# 三种不同参数的滤波器
configs = [
    {'Q': 0.001, 'R': 0.01, 'label': 'Q=0.001, R=0.01 (项目默认)', 'color': '#e74c3c'},
    {'Q': 0.01,  'R': 0.01, 'label': 'Q=0.01, R=0.01 (Q/R=1)', 'color': '#3498db'},
    {'Q': 0.001, 'R': 0.1,  'label': 'Q=0.001, R=0.1 (强平滑)', 'color': '#2ecc71'},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, cfg in zip(axes, configs):
    kf = KalmanFilter1D(Q=cfg['Q'], R=cfg['R'])
    filtered = [kf.update(z) for z in noisy]
    
    ax.plot(t, noisy, alpha=0.3, color='gray', label='Raw + noise')
    ax.plot(t, true_signal, '--', color='black', linewidth=1, label='Ground truth')
    ax.plot(t, filtered, color=cfg['color'], linewidth=1.5, label=cfg['label'])
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Sensor value')
    ax.legend(fontsize=8)
    ax.set_title(f'Q={cfg["Q"]}, R={cfg["R"]}, Q/R={cfg["Q"]/cfg["R"]}')

plt.suptitle('Kalman Filter 1D — Parameter Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- 卡尔曼增益收敛可视化 ----
kf = KalmanFilter1D(Q=0.001, R=0.01)
gains = []
for z in noisy:
    # 手动计算 K 用于记录
    P_pred = kf.P + kf.Q
    K = P_pred / (P_pred + kf.R)
    gains.append(K)
    kf.update(z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(gains, color='#e74c3c')
ax1.set_xlabel('Sample index')
ax1.set_ylabel('Kalman Gain K')
ax1.set_title('Kalman Gain Convergence')
ax1.axhline(y=kf.Q/(kf.Q+kf.R), color='gray', linestyle='--', 
            label=f'Steady state K≈{kf.Q/(kf.Q+kf.R):.4f}')
ax1.legend()

# P 收敛
kf2 = KalmanFilter1D(Q=0.001, R=0.01)
P_trace = []
for z in noisy:
    P_trace.append(kf2.P)
    kf2.update(z)

ax2.semilogy(P_trace, color='#3498db')
ax2.set_xlabel('Sample index')
ax2.set_ylabel('Covariance P (log scale)')
ax2.set_title('Estimate Covariance Convergence')

plt.tight_layout()
plt.show()

print(f'稳态卡尔曼增益 K_ss ≈ {gains[-1]:.6f}')
print(f'稳态协方差 P_ss ≈ {P_trace[-1]:.6f}')

## 4. Min-Max 特征归一化

### 4.1 算法

对每个特征通道 $i$，在**校准阶段**（200帧 = 2秒）记录运行极值：

$$
\text{min}_i = \min_{k=1}^{N_{\text{cal}}} x_i^{(k)}, \quad
\text{max}_i = \max_{k=1}^{N_{\text{cal}}} x_i^{(k)}
$$

在**推理阶段**，归一化公式为：

$$
\hat{x}_i = \text{clamp}\left(\frac{x_i - \text{min}_i}{\text{max}_i - \text{min}_i},\ 0,\ 1\right)
$$

当 $\text{max}_i = \text{min}_i$（零方差通道）时：

$$
\hat{x}_i = 0.5
$$

### 4.2 特点

- 校准是**每设备、每次启动**独立进行的
- min/max 不跨重启持久化
- 输出范围严格 $[0, 1]$，适合 TFLite 量化

In [ ]:
class FeatureNormalizer:
    """
    Min-Max 归一化器，对应固件 lib/Filters/FeatureNormalizer.h。
    
    两个阶段：
    1. 校准阶段 (calibration): 记录每通道 min/max
    2. 推理阶段 (inference): 归一化到 [0, 1]
    """
    
    def __init__(self, num_features: int = FEATURE_COUNT, 
                 calibration_frames: int = 200):
        self.num_features = num_features
        self.calibration_frames = calibration_frames
        self.feat_min = np.full(num_features, np.inf)
        self.feat_max = np.full(num_features, -np.inf)
        self.frame_count = 0
        self._calibrated = False
    
    def update_stats(self, x: np.ndarray) -> None:
        """校准阶段：更新每通道 min/max。"""
        self.feat_min = np.minimum(self.feat_min, x)
        self.feat_max = np.maximum(self.feat_max, x)
        self.frame_count += 1
        if self.frame_count >= self.calibration_frames:
            self._calibrated = True
    
    def normalize(self, x: np.ndarray) -> np.ndarray:
        """推理阶段：归一化到 [0, 1]。"""
        rng = self.feat_max - self.feat_min
        # 零方差通道映射到 0.5
        result = np.where(
            rng > 0,
            np.clip((x - self.feat_min) / rng, 0.0, 1.0),
            0.5
        )
        return result
    
    @property
    def is_calibrated(self) -> bool:
        return self._calibrated


# ---- 可视化归一化效果 ----
np.random.seed(42)

# 模拟 21 通道传感器数据（不同量纲、不同范围）
channel_ranges = {
    'Hall sensors (mT)': (-40, 40),
    'Euler angles (deg)': (-180, 180),
    'Gyro (deg/s)': (-500, 500),
}

raw_data = np.zeros((300, FEATURE_COUNT))
# Hall: 15 channels, ±40 mT range
raw_data[:, :15] = np.random.normal(0, 10, (300, 15))
# Euler: 3 channels
raw_data[:, 15:18] = np.random.normal(0, 30, (300, 3))
# Gyro: 3 channels
raw_data[:, 18:21] = np.random.normal(0, 100, (300, 3))

normalizer = FeatureNormalizer()

# 校准阶段
for i in range(200):
    normalizer.update_stats(raw_data[i])

# 归一化
normalized = np.array([normalizer.normalize(raw_data[i]) for i in range(300)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 原始数据
ax1.plot(raw_data[:, 0], label='hall_0_x (mT)', alpha=0.7)
ax1.plot(raw_data[:, 15], label='euler_roll (deg)', alpha=0.7)
ax1.plot(raw_data[:, 18], label='gyro_x (deg/s)', alpha=0.7)
ax1.set_title('Raw Sensor Data (Different Scales)')
ax1.set_xlabel('Frame')
ax1.set_ylabel('Value (original units)')
ax1.legend()
ax1.axvline(x=200, color='red', linestyle='--', alpha=0.5, label='Calibration end')

# 归一化后
ax2.plot(normalized[:, 0], label='hall_0_x', alpha=0.7)
ax2.plot(normalized[:, 15], label='euler_roll', alpha=0.7)
ax2.plot(normalized[:, 18], label='gyro_x', alpha=0.7)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax2.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
ax2.set_title('After Min-Max Normalization [0, 1]')
ax2.set_xlabel('Frame')
ax2.set_ylabel('Normalized value')
ax2.legend()
ax2.axvline(x=200, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f'Calibration range (hall_0_x): [{normalizer.feat_min[0]:.2f}, {normalizer.feat_max[0]:.2f}] mT')
print(f'Calibration range (euler_roll): [{normalizer.feat_min[15]:.2f}, {normalizer.feat_max[15]:.2f}] deg')
print(f'Calibration range (gyro_x): [{normalizer.feat_min[18]:.2f}, {normalizer.feat_max[18]:.2f}] deg/s')

## 5. 滑动窗口（Sliding Window）

### 5.1 数据结构

滑动窗口使用**环形缓冲区（Ring Buffer）**实现：

$$
\mathbf{X} = \begin{bmatrix} \mathbf{x}_1^T \\ \mathbf{x}_2^T \\ \vdots \\ \mathbf{x}_{30}^T \end{bmatrix} \in \mathbb{R}^{30 \times 21}
$$

- 内存布局：连续 `float[30 × 21] = 630` 个浮点数
- 物理大小：630 × 4 bytes = **2,520 bytes**
- 分配在 PSRAM 中（ESP32-S3 N16R8）
- **SPSC**（单生产者单消费者）设计，无需互斥锁

### 5.2 时序意义

在 100 Hz 采样率下：

$$
\Delta t = \frac{1}{100} = 10\ \text{ms}, \quad T_{\text{window}} = 30 \times 10 = 300\ \text{ms}
$$

300ms 的窗口足以覆盖大多数手语手势的完整动作周期。

In [ ]:
class SlidingWindow:
    """
    环形缓冲区滑动窗口，对应固件 lib/Filters/SlidingWindow.h。
    
    Parameters
    ----------
    window_size : int
        窗口帧数 (default 30)
    feature_dim : int
        每帧特征维度 (default 21)
    """
    
    def __init__(self, window_size: int = WINDOW_SIZE, 
                 feature_dim: int = FEATURE_COUNT):
        self.window_size = window_size
        self.feature_dim = feature_dim
        # 环形缓冲区
        self._buffer = np.zeros((window_size, feature_dim), dtype=np.float32)
        self._write_idx = 0
        self._count = 0
    
    def push(self, frame: np.ndarray) -> None:
        """推入新的一帧数据。"""
        self._buffer[self._write_idx] = frame
        self._write_idx = (self._write_idx + 1) % self.window_size
        self._count = min(self._count + 1, self.window_size)
    
    def get_window(self) -> np.ndarray:
        """
        获取当前窗口数据（按时间顺序排列）。
        
        Returns
        -------
        np.ndarray
            Shape (window_size, feature_dim)，最旧帧在前
        """
        if self._count < self.window_size:
            # 未满：返回有效部分
            return self._buffer[:self._count].copy()
        # 已满：从写指针开始重新排列
        idx = self._write_idx
        return np.concatenate([
            self._buffer[idx:],   # 旧数据
            self._buffer[:idx],   # 新数据
        ])
    
    @property
    def is_full(self) -> bool:
        return self._count >= self.window_size
    
    def reset(self):
        """清空窗口。"""
        self._buffer[:] = 0
        self._write_idx = 0
        self._count = 0


# ---- 可视化：滑动窗口填充过程 ----
window = SlidingWindow()

# 模拟一个简单的手势：先弯曲再伸展
frames = []
for t in range(60):
    if t < 15:
        # 弯曲阶段
        frame = np.zeros(FEATURE_COUNT)
        frame[0] = 5.0 * (t / 15)   # hall_0_x 递增
        frame[15] = 30.0 * (t / 15)  # euler_roll 递增
    elif t < 30:
        # 保持
        frame = np.zeros(FEATURE_COUNT)
        frame[0] = 5.0
        frame[15] = 30.0
    else:
        # 伸展阶段
        progress = (t - 30) / 30
        frame = np.zeros(FEATURE_COUNT)
        frame[0] = 5.0 * (1 - progress)
        frame[15] = 30.0 * (1 - progress)
    window.push(frame)
    frames.append(frame.copy())

frames = np.array(frames)

# 可视化窗口内容在不同时间点的状态
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
snapshots = [10, 20, 30, 40, 50, 59]

for ax, snap_idx in zip(axes.flat, snapshots):
    # 重建该时刻的窗口状态
    w = SlidingWindow()
    for i in range(snap_idx + 1):
        w.push(frames[i])
    
    win_data = w.get_window()
    ax.plot(win_data[:, 0], label='hall_0_x', linewidth=2)
    ax.plot(win_data[:, 15], label='euler_roll', linewidth=2)
    ax.set_title(f'Frame {snap_idx} (window {len(win_data)}/30)')
    ax.set_xlabel('Window frame index')
    ax.set_ylabel('Value')
    ax.legend(fontsize=8)
    ax.set_ylim(-1, 35)

plt.suptitle('Sliding Window Content at Different Time Steps', fontsize=13)
plt.tight_layout()
plt.show()

## 6. 合成数据生成

由于硬件尚未完全就绪，使用**合成数据**进行开发和验证。每个手势通过参数化的手部姿态模型生成 21 通道时序数据。

### 6.1 合成策略

对于每个手势类别 $c$，定义一个**基线姿态向量** $\mathbf{b}_c \in \mathbb{R}^{21}$，然后叠加时序动态：

$$
\mathbf{x}_t^{(c)} = \mathbf{b}_c + A_c \sin(2\pi f_c t + \boldsymbol{\varphi}_c) + \boldsymbol{\epsilon}_t
$$

其中：
- $A_c$: 振幅向量（模拟手势动作幅度）
- $f_c$: 动作频率（Hz）
- $\boldsymbol{\varphi}_c$: 各通道相位偏移
- $\boldsymbol{\epsilon}_t \sim \mathcal{N}(0, \sigma^2)$: 测量噪声

In [ ]:
def generate_synthetic_gesture(
    gesture_id: int,
    num_frames: int = 30,
    sample_rate: int = 100,
    noise_std: float = 0.5,
    seed: int = None,
) -> np.ndarray:
    """
    为单个手势生成合成传感器数据。
    
    Parameters
    ----------
    gesture_id : int
        手势类别 ID (0-45)
    num_frames : int
        帧数
    sample_rate : int
        采样率 (Hz)
    noise_std : float
        噪声标准差
    seed : int, optional
        随机种子
        
    Returns
    -------
    np.ndarray
        Shape (num_frames, 21)
    """
    rng = np.random.default_rng(seed if seed is not None else gesture_id * 1000)
    dt = 1.0 / sample_rate
    t = np.arange(num_frames) * dt
    
    # 基线姿态：不同手势有不同的传感器基线
    np.random.seed(gesture_id * 42)
    baseline = np.zeros(FEATURE_COUNT)
    
    # Hall sensors: 根据手势ID生成不同的弯曲模式
    for i in range(NUM_HALL_SENSORS):
        # 每个手指的弯曲程度由手势ID决定
        bend_factor = np.sin(gesture_id * 0.7 + i * 1.2) * 15
        baseline[i*3:(i+1)*3] = [bend_factor, bend_factor * 0.3, bend_factor * 0.1]
    
    # Euler angles
    baseline[15] = np.sin(gesture_id * 0.5) * 20   # roll
    baseline[16] = np.cos(gesture_id * 0.3) * 15   # pitch
    baseline[17] = gesture_id * 2.0 % 60 - 30      # yaw
    
    # Gyro: 动态部分
    baseline[18:21] = rng.normal(0, 10, 3)
    
    # 时序动态（手势动作）
    freq = 1.0 + (gesture_id % 5) * 0.5  # 1.0-3.5 Hz
    amplitude = np.zeros(FEATURE_COUNT)
    amplitude[:15] = rng.uniform(2, 8, 15)    # Hall 动态幅度
    amplitude[15:18] = rng.uniform(5, 20, 3)  # Euler 动态幅度
    amplitude[18:21] = rng.uniform(30, 100, 3) # Gyro 动态幅度
    
    # 各通道相位偏移
    phases = rng.uniform(0, 2 * np.pi, FEATURE_COUNT)
    
    # 生成信号
    signal = np.outer(np.ones(num_frames), baseline)
    for ch in range(FEATURE_COUNT):
        signal[:, ch] += amplitude[ch] * np.sin(2 * np.pi * freq * t + phases[ch])
    
    # 添加噪声
    noise = rng.normal(0, noise_std, (num_frames, FEATURE_COUNT))
    
    return signal + noise


def generate_dataset(
    num_classes: int = 20,
    samples_per_class: int = 50,
    num_frames: int = 30,
    add_kalman: bool = True,
    add_normalize: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    生成完整合成数据集。
    
    Returns
    -------
    X : np.ndarray
        Shape (N, num_frames, 21) — 滑动窗口数据
    y : np.ndarray
        Shape (N,) — 类别标签
    """
    X_list = []
    y_list = []
    
    for cls in range(num_classes):
        for s in range(samples_per_class):
            # 生成原始数据（多取一些帧用于卡尔曼滤波预热）
            raw = generate_synthetic_gesture(
                gesture_id=cls,
                num_frames=num_frames + 10,
                seed=cls * 1000 + s,
            )
            
            if add_kalman:
                # 应用卡尔曼滤波
                filtered = np.zeros_like(raw)
                for ch in range(FEATURE_COUNT):
                    kf = KalmanFilter1D(Q=0.001, R=0.01)
                    for t in range(len(raw)):
                        filtered[t, ch] = kf.update(raw[t, ch])
                # 跳过预热帧
                raw = filtered[10:]
            
            X_list.append(raw)
            y_list.append(cls)
    
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    
    if add_normalize:
        # 全局归一化（训练时用 fit-once 策略）
        all_flat = X.reshape(-1, FEATURE_COUNT)
        feat_min = all_flat.min(axis=0)
        feat_max = all_flat.max(axis=0)
        rng = feat_max - feat_min
        rng[rng == 0] = 1  # 避免除零
        X = (X - feat_min) / rng
        X = np.clip(X, 0, 1)
    
    return X, y


# ---- 生成数据集 ----
X, y = generate_dataset(num_classes=20, samples_per_class=50, num_frames=30)
print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'X range: [{X.min():.4f}, {X.max():.4f}]')
print(f'Classes: {np.unique(y)}')

In [ ]:
# ---- 可视化不同手势的特征模式 ----
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

gesture_ids = [0, 1, 10, 14, 15, 26, 43, 45]
for ax, gid in zip(axes.flat, gesture_ids):
    # 取该类别的第一个样本
    idx = np.where(y == gid)[0]
    if len(idx) == 0:
        continue
    sample = X[idx[0]]  # (30, 21)
    
    ax.plot(sample[:, 0], label='hall_0_x', alpha=0.8)
    ax.plot(sample[:, 5], label='hall_1_x', alpha=0.8)
    ax.plot(sample[:, 15], label='euler_roll', alpha=0.8)
    
    label_info = next((g for g in GESTURE_LABELS if g['id'] == gid), None)
    title = f'[{gid}] {label_info["name_cn"]} ({label_info["name_en"]})' if label_info else f'Class {gid}'
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Frame')
    ax.legend(fontsize=7)

plt.suptitle('Synthetic Gesture Patterns (Normalized)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Edge Impulse 数据转发器格式

### 7.1 CSV 格式

固件通过串口输出的 CSV 格式兼容 Edge Impulse 数据转发器：

```csv
hall_0_x,hall_0_y,hall_0_z,hall_1_x,...,euler_roll,euler_pitch,euler_yaw,gyro_x,gyro_y,gyro_z
-1.234,0.567,2.891,...,10.5,-5.3,0.2,15.2,-8.1,3.4
...
```

### 7.2 采集流程

1. 固件以 100Hz 输出 CSV 到串口
2. 运行 `edge-impulse-data-forwarder`（本地工具）
3. 工具将串口数据转发到 Edge Impulse 平台
4. 在 Edge Impulse 中标注、训练、导出模型

### 7.3 生成可导入的 CSV 文件

In [ ]:
import csv
import io


def export_to_csv(X: np.ndarray, y: np.ndarray, 
                  output_dir: str = 'data/edge_impulse') -> None:
    """
    将数据集导出为 Edge Impulse 兼容的 CSV 文件。
    每个类别一个文件，格式：label_timestamp.csv
    
    Parameters
    ----------
    X : np.ndarray
        Shape (N, T, 21)
    y : np.ndarray
        Shape (N,)
    output_dir : str
        输出目录
    """
    os.makedirs(output_dir, exist_ok=True)
    
    header = FEATURE_NAMES
    
    for cls_id in np.unique(y):
        cls_mask = y == cls_id
        cls_data = X[cls_mask]
        
        label_info = next((g for g in GESTURE_LABELS if g['id'] == cls_id), None)
        label_name = label_info['name_en'].replace(' ', '_') if label_info else f'class_{cls_id}'
        
        for sample_idx, sample in enumerate(cls_data):
            filename = f'{label_name}_{sample_idx:04d}.csv'
            filepath = os.path.join(output_dir, filename)
            
            with open(filepath, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(header)
                for frame in sample:
                    writer.writerow(frame.tolist())
    
    total_files = len(np.unique(y)) * cls_data.shape[0] if len(np.unique(y)) > 0 else 0
    print(f'Exported {total_files} CSV files to {output_dir}/')
    print(f'  Classes: {len(np.unique(y))}')
    print(f'  Frames per file: {X.shape[1]}')
    print(f'  Features per frame: {X.shape[2]}')


# 导出示例（只导出前 3 个类别各 2 个样本）
X_small = X[:30]  # 前 30 个样本
y_small = y[:30]
export_to_csv(X_small, y_small, output_dir=str(PROJECT_ROOT / 'data' / 'edge_impulse_sample'))

# 打印一个示例文件内容
sample_files = list((PROJECT_ROOT / 'data' / 'edge_impulse_sample').glob('*.csv'))
if sample_files:
    print(f'\n--- Sample: {sample_files[0].name} ---')
    with open(sample_files[0]) as f:
        print(f.read()[:500])

## 8. 数据质量验证

### 8.1 验证检查项

| 检查项 | 标准 |
|--------|------|
| 采样率稳定性 | ±1% (99-101 Hz) |
| 数据完整性 | 无丢帧（seq 连续） |
| 范围有效性 | Hall: ±40 mT, Euler: ±180°, Gyro: ±500°/s |
| 类别均衡 | 每类样本数差异 < 20% |
| 噪声水平 | SNR > 20 dB |

In [ ]:
def validate_dataset(X: np.ndarray, y: np.ndarray) -> dict:
    """
    数据集质量验证。
    
    Returns
    -------
    dict
        验证结果报告
    """
    report = {}
    
    # 1. 基本统计
    report['total_samples'] = len(y)
    report['num_classes'] = len(np.unique(y))
    report['frames_per_sample'] = X.shape[1]
    report['features_per_frame'] = X.shape[2]
    
    # 2. 数值范围
    report['value_range'] = (float(X.min()), float(X.max()))
    report['has_nan'] = bool(np.isnan(X).any())
    report['has_inf'] = bool(np.isinf(X).any())
    
    # 3. 类别分布
    unique, counts = np.unique(y, return_counts=True)
    report['class_counts'] = dict(zip(unique.tolist(), counts.tolist()))
    report['class_balance'] = float(counts.min() / counts.max())
    
    # 4. 每通道统计
    report['per_channel_mean'] = X.mean(axis=(0, 1)).tolist()
    report['per_channel_std'] = X.std(axis=(0, 1)).tolist()
    
    return report


report = validate_dataset(X, y)

print('=' * 60)
print('Dataset Validation Report')
print('=' * 60)
print(f'Total samples: {report["total_samples"]}')
print(f'Num classes: {report["num_classes"]}')
print(f'Frames/sample: {report["frames_per_sample"]}')
print(f'Features/frame: {report["features_per_frame"]}')
print(f'Value range: [{report["value_range"][0]:.4f}, {report["value_range"][1]:.4f}]')
print(f'Has NaN: {report["has_nan"]}')
print(f'Has Inf: {report["has_inf"]}')
print(f'Class balance ratio: {report["class_balance"]:.2%}')
print(f'Per-channel std range: [{min(report["per_channel_std"]):.4f}, {max(report["per_channel_std"]):.4f}]')

In [ ]:
# ---- 类别分布可视化 ----
unique, counts = np.unique(y, return_counts=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 条形图
bars = ax1.bar(unique, counts, color='#3498db', alpha=0.8)
ax1.set_xlabel('Gesture Class ID')
ax1.set_ylabel('Sample Count')
ax1.set_title('Class Distribution')
ax1.axhline(y=counts.mean(), color='red', linestyle='--', alpha=0.5, 
            label=f'Mean = {counts.mean():.0f}')
ax1.legend()

# 通道标准差热力图
per_class_std = np.zeros((len(unique), FEATURE_COUNT))
for cls in unique:
    cls_data = X[y == cls]
    per_class_std[cls] = cls_data.std(axis=(0, 1))

im = ax2.imshow(per_class_std, aspect='auto', cmap='YlOrRd')
ax2.set_xlabel('Feature Channel')
ax2.set_ylabel('Gesture Class')
ax2.set_title('Per-Class Per-Channel Std (Variability)')
plt.colorbar(im, ax=ax2, label='Std')

plt.tight_layout()
plt.show()

## 9. 完整信号处理流水线

将上述组件组合为完整的流水线，对应固件的 `SensorManager → Kalman → Normalize → SlidingWindow → CSV` 链路。

```
Raw Sensor → KalmanFilter1D[21] → FeatureNormalizer → SlidingWindow(30) → Model Input
  (mT,°,°/s)      (denoise)         ([0,1])          (30×21=630)      (B,30,21)
```

In [ ]:
class SignalProcessingPipeline:
    """
    完整的信号处理流水线。
    对应固件中 Task_SensorRead 的完整处理链路。
    """
    
    def __init__(self, feature_count: int = FEATURE_COUNT,
                 window_size: int = WINDOW_SIZE,
                 calibration_frames: int = 200,
                 kalman_Q: float = 0.001,
                 kalman_R: float = 0.01):
        self.feature_count = feature_count
        self.window_size = window_size
        
        # 21 个独立卡尔曼滤波器
        self.kalman_filters = [KalmanFilter1D(Q=kalman_Q, R=kalman_R) 
                               for _ in range(feature_count)]
        # 归一化器
        self.normalizer = FeatureNormalizer(feature_count, calibration_frames)
        # 滑动窗口
        self.window = SlidingWindow(window_size, feature_count)
        
        self._frame_count = 0
    
    def process_frame(self, raw_frame: np.ndarray) -> Tuple[bool, np.ndarray]:
        """
        处理一帧原始传感器数据。
        
        Parameters
        ----------
        raw_frame : np.ndarray
            Shape (21,) — 原始传感器读数
            
        Returns
        -------
        is_ready : bool
            窗口是否已满（可以推理）
        window_data : np.ndarray
            Shape (30, 21) if ready, else empty
        """
        # Step 1: 卡尔曼滤波
        filtered = np.array([
            self.kalman_filters[ch].update(raw_frame[ch])
            for ch in range(self.feature_count)
        ])
        
        # Step 2: 归一化
        if not self.normalizer.is_calibrated:
            self.normalizer.update_stats(filtered)
            normalized = filtered  # 校准阶段不归一化
        else:
            normalized = self.normalizer.normalize(filtered)
        
        # Step 3: 推入滑动窗口
        self.window.push(normalized)
        self._frame_count += 1
        
        # 返回窗口数据
        if self.window.is_full:
            return True, self.window.get_window()
        return False, np.array([])


# ---- 端到端测试 ----
pipeline = SignalProcessingPipeline()

# 模拟 300 帧原始传感器数据
np.random.seed(42)
raw_stream = np.random.randn(300, FEATURE_COUNT) * 10  # 原始量级

# 不同通道不同量纲
raw_stream[:, :15] *= 4   # Hall: ±40 mT
raw_stream[:, 15:18] *= 18  # Euler: ±180 deg
raw_stream[:, 18:21] *= 50  # Gyro: ±500 deg/s

ready_count = 0
for i in range(300):
    is_ready, window_data = pipeline.process_frame(raw_stream[i])
    if is_ready:
        ready_count += 1

print(f'Frames processed: 300')
print(f'Calibration phase: {pipeline.normalizer.calibration_frames} frames')
print(f'Window ready count: {ready_count}')
print(f'Window shape when ready: {window_data.shape}')
print(f'Window value range: [{window_data.min():.4f}, {window_data.max():.4f}]')

## 10. 保存预处理数据集

将生成的合成数据集保存为 NumPy `.npz` 格式，供后续训练 Notebooks 使用。

In [ ]:
# ---- 生成并保存完整数据集 ----
print('Generating full synthetic dataset...')
print('  20 classes × 100 samples × 30 frames × 21 features')

X_full, y_full = generate_dataset(
    num_classes=20,
    samples_per_class=100,
    num_frames=30,
    add_kalman=True,
    add_normalize=True,
)

# 划分训练集/验证集 (80/20)
np.random.seed(42)
indices = np.random.permutation(len(y_full))
split = int(0.8 * len(y_full))
train_idx, val_idx = indices[:split], indices[split:]

X_train, y_train = X_full[train_idx], y_full[train_idx]
X_val, y_val = X_full[val_idx], y_full[val_idx]

# 保存
output_dir = PROJECT_ROOT / 'data' / 'processed'
os.makedirs(output_dir, exist_ok=True)

np.savez(
    output_dir / 'synthetic_dataset.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    feature_names=FEATURE_NAMES,
    num_classes=20,
    window_size=WINDOW_SIZE,
    feature_count=FEATURE_COUNT,
)

print(f'\nSaved to {output_dir / "synthetic_dataset.npz"}')
print(f'  X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape}  y_val:   {y_val.shape}')
print(f'  File size: {(output_dir / "synthetic_dataset.npz").stat().st_size / 1024 / 1024:.2f} MB')

---

## 小结

| 组件 | 算法 | 关键参数 |
|------|------|----------|
| 卡尔曼滤波 | 一维恒值模型 | Q=0.001, R=0.01 |
| 归一化 | Min-Max [0, 1] | 校准 200 帧 |
| 滑动窗口 | 环形缓冲区 | 30帧 × 21特征 = 630 floats |
| 合成数据 | 参数化正弦叠加 | 20类 × 100样本 |
| CSV导出 | Edge Impulse 格式 | 每帧一行，21列 |

**下一步**: 运行 `02_l1_model_training.ipynb` 进行 L1 模型训练。